# Planewave DFT with Abinit and AbiPy -- Part 1: Running calculations

**CEMDI 2026 -- hands-on session**

This is the first of two notebooks that make up the companion material for
the 3-hour CEMDI tutorial on planewave density-functional theory (DFT) with
[Abinit](https://www.abinit.org), using [AbiPy](https://github.com/abinit/abipy)
throughout -- a Python framework built on top of `pymatgen` that generates
Abinit input files, submits and monitors calculations organized as
`Flows`/`Works`/`Tasks`, and post-processes the native netcdf output files
(`GSR.nc`, `DDB`, ...).

This notebook (**Part 1**) is about *building and launching* calculations:
`AbinitInput` objects and the `Flow`s that wrap them. The companion
notebook, [`2_analysis_and_plotting.ipynb`](2_analysis_and_plotting.ipynb)
(**Part 2**), picks up from there and covers analysis and plotting --
including AbiPy `Robot`s and the command-line tools, which are only
introduced in that notebook's last section.

The style and many of the code patterns below follow the
[AbiPy Jupyter Book](https://github.com/abinit/abipy_book) lessons -- in
particular `base3/lesson_base3.md` (silicon) and `dfpt/lesson_dfpt.md`
(phonons of AlAs). All the flow-building functions we call are collected in
`workshop_lib.py`, next to this notebook; standalone equivalents that you
can run from a terminal, without this notebook, live in `../Examples/`.

We work with a single test system throughout most of the notebook: gallium
arsenide (GaAs), a simple, fast-converging III-V zinc-blende semiconductor.
Silicon is added once, in the band-structure section, for comparison.

**Outline**

1. Setup
2. Building an `AbinitInput`
3. Ground-state total energy of GaAs
4. Convergence studies (ecut, k-points)
5. Band structures: GaAs vs Si
6. Equation of state and the lattice parameter
7. Phonons from DFPT


**Note.** Sections 6 and 7 (equation of state and phonons) are new material
written for this workshop -- there was no ready-made example for them in
`AbipyExamples` yet. The flows they build are deliberately coarse (small
`ecut`, small q-mesh) to keep them cheap; treat the numbers you get as
illustrative rather than converged.

## 1. Setup

Before generating any input, make sure Abinit is on your `$PATH` and that
`~/.abinit/abipy/manager.yml` and `scheduler.yml` are configured for your
machine (templates are available in
`AbipyExamples/Data/Config_files/config_local`). You can check your
installation with:

```
abicheck.py
```

Every `build_*_flow()` function below also has a standalone counterpart in
`../Examples/` (e.g. `run_gaas_gstate.py` for Section 3) that builds and
launches the same flow from a terminal, without a notebook -- this is the
recommended way to run anything non-trivial, since it doesn't block the
notebook kernel. Run them from this `Notebooks/` directory so that the
`flow_*/` folders they create land next to this notebook, matching the
paths used throughout:

```
python ../Examples/run_gaas_gstate.py
abirun.py flow_gaas_gstate scheduler
```

In [ ]:
import warnings
warnings.filterwarnings("ignore")  # Ignore warnings for the tutorial

import numpy as np

from abipy import abilab
abilab.enable_notebook()  # Tell AbiPy we are running inside a notebook

# Show matplotlib figures embedded in the notebook.
%matplotlib inline

import workshop_lib as wlib

`workshop_lib` gives us one `build_*_flow()` function per section below,
plus `gaas_structure()` / `si_structure()` and a shared `setup_manager()`.
Let's have a first look at the GaAs structure:

In [ ]:
structure = wlib.gaas_structure()
print(structure)

In [ ]:
structure.plot();

## 2. Building an `AbinitInput`

An `AbinitInput` is a dict-like object: it *has* a `Structure`, a list of
`Pseudo` objects, and a set of Abinit input variables. `workshop_lib.gs_input`
builds one for a GaAs ground-state (GS) run -- let's look at the source:

In [ ]:
abilab.print_source(wlib.gs_input)

In [ ]:
gsinp = wlib.gs_input(ecut=6, ngkpt=(8, 8, 8))
print("ecut:", gsinp["ecut"], "Ha")
print("ngkpt:", gsinp["ngkpt"])
gsinp

> **Note.** Indices in Python (and in AbiPy) start at 0: the first band has
> index 0, the first spin channel is 0, etc. Keep this in mind whenever you
> index bands, k-points or spins.

The pseudopotentials come from [PseudoDojo](http://www.pseudo-dojo.org)
(norm-conserving, scalar-relativistic, PBE, in `Data/Pseudos`):

In [ ]:
for pseudo in gsinp.pseudos:
    print(pseudo)

## 3. Ground-state total energy of GaAs

`build_gs_flow` wraps `gs_input` in a `Flow` with a single SCF task
-- the standalone version is `Examples/run_gaas_gstate.py`:

In [ ]:
abilab.print_source(wlib.build_gs_flow)

In [ ]:
flow = wlib.build_gs_flow()
#flow.get_graphviz()

To actually run it, either call `flow.make_scheduler().start()` here in the
notebook, or (recommended for anything non-trivial) run the standalone
script from a terminal, as described in Section 1:

```
python ../Examples/run_gaas_gstate.py
abirun.py flow_gaas_gstate scheduler
```

This particular flow has already been run for you (`flow_gaas_gstate/`,
next to this notebook). Head to
[`2_analysis_and_plotting.ipynb`](2_analysis_and_plotting.ipynb) to look at
the results, or keep going below to build the remaining flows.


## 4. Convergence studies

Two Abinit parameters control the accuracy of a planewave DFT calculation
for a given pseudopotential set: the plane-wave cutoff `ecut` and the
density of the k-point mesh. AbiPy makes it easy to launch a family of GS
runs that only differ by one parameter; we'll collect and compare the
results with a `Robot` in Part 2.

### 4.1 ecut convergence

`build_ecut_conv_flow` registers one SCF task per value of `ecut`
-- standalone version: `Examples/run_gaas_convecut.py`:

In [ ]:
abilab.print_source(wlib.build_ecut_conv_flow)

In [ ]:
flow = wlib.build_ecut_conv_flow()
print(f"{len(list(flow.iflat_tasks()))} SCF tasks, ecut = "
      f"{[t.input['ecut'] for t in flow.iflat_tasks()]}")

This flow -- and the k-point convergence flow below -- have already been
run for you (`flow_gaas_convecut/`, `flow_gaas_convkpt/`). We'll compare
the results with a `GsrRobot` in Part 2's last section.


### 4.2 k-point convergence

Same idea, but now `ecut` is fixed and we vary the k-mesh via
`set_autokmesh(nk)`, which generates an increasingly dense homogeneous mesh
-- standalone version: `Examples/run_gaas_convkpt.py`:

In [ ]:
abilab.print_source(wlib.build_kpt_conv_flow)

## 5. Band structures: GaAs vs Si

A band-structure flow chains two datasets: a GS run on a homogeneous k-mesh
(to get the density), then a non-self-consistent (NSCF) run along a
high-symmetry k-path using that density. `flowtk.bandstructure_flow` builds
this two-task work for us; `build_gaas_ebands_flow` /
`build_si_ebands_flow` wrap it for each material
-- standalone versions: `Examples/run_gaas_ebands.py`,
`Examples/run_si_ebands.py`:

In [ ]:
abilab.print_source(wlib.build_gaas_ebands_flow)

In [ ]:
flow_gaas = wlib.build_gaas_ebands_flow()
flow_si = wlib.build_si_ebands_flow()

Both flows have already been run for you (`flow_gaas_ebands/`,
`flow_si_ebands/`). Head to
[`2_analysis_and_plotting.ipynb`](2_analysis_and_plotting.ipynb) to plot
the resulting band structures, or keep going below.


## 6. Equation of state and the lattice parameter

Rather than running a full ionic + cell relaxation (`ionmov`/`optcell`),
which can be finicky to converge live in a 3h session, we use the
equation-of-state (EOS) approach: run the SCF ground state at several
isotropically-scaled volumes around the experimental structure, then fit a
Birch-Murnaghan equation of state to $E(V)$. This is the same strategy used
in the AbiPy `base3` (silicon) lesson to determine the lattice parameter.

`build_eos_flow` builds one SCF task per scaled volume:

In [ ]:
abilab.print_source(wlib.build_eos_flow)

In [ ]:
flow = wlib.build_eos_flow()

This flow has already been run for you (`flow_gaas_eos/`) -- standalone
version: `Examples/run_gaas_eos.py`. The Birch-Murnaghan fit itself needs a
`GsrRobot` to gather all the scaled-volume results together, so we leave it
for Part 2's last section.


## 7. Phonons from DFPT

Density-functional perturbation theory (DFPT) gives access to phonons,
Born effective charges and the dielectric tensor without finite
displacements. AbiPy's `PhononFlow` automates the whole workflow: a GS run
that produces the ground-state wavefunctions (`WFK`), followed by one DFPT
task per symmetry-irreducible atomic perturbation and q-point.

`build_phonon_flow` is a simplified, GaAs version -- coarser `ecut`,
k-mesh and q-mesh, for speed -- of a production DFPT phonon
workflow; standalone version: `Examples/run_gaas_phonons.py`:

In [ ]:
abilab.print_source(wlib.build_phonon_flow)

In [ ]:
flow = wlib.build_phonon_flow()
#flow.get_graphviz()

This flow has more tasks than the others (one per symmetry-irreducible
atomic perturbation and q-point), so it's the best candidate for actually
running with `nohup python ../Examples/run_gaas_phonons.py &` in the
background rather than waiting on it here. It has already been run for you
(`flow_gaas_phonons/`) -- head to
[`2_analysis_and_plotting.ipynb`](2_analysis_and_plotting.ipynb) for the
resulting phonon bands, DOS and Born effective charges.


## End of Part 1

All the flows used in this tutorial have now been built (and, behind the
scenes, already run for you): ground-state energy, ecut/k-point
convergence, band structures for GaAs and Si, an equation-of-state series,
and a DFPT phonon calculation.

Continue with
[`2_analysis_and_plotting.ipynb`](2_analysis_and_plotting.ipynb) to look at
the results.
